[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/YOUR-GITHUB-USERNAME/JAXCode/blob/master/solutions/b_27_cross_attention_pure_solution.ipynb)

# 🟡 Solution: Cross-Attention without Flax

*Attention & Transformers · Medium*

Reference implementation. Try it yourself in `b_27_cross_attention_pure.ipynb` first.

---
Problem 23 with an explicit parameter pytree.

### Signature
```python
def init_cross_attention(key, d_model, num_heads):
    ...   # -> {"W_q": {...}, "W_k": {...}, "W_v": {...}, "W_o": {...}}

def apply_cross_attention(params, x_q, x_kv, num_heads):
    ...   # (B, seq_q, d_model), (B, seq_kv, d_model) -> (B, seq_q, d_model)
```

Same pytree as `b_26`: four `(d_model, d_model)` kernels scaled by
`1/sqrt(d_model)`, four zero biases, key split four ways.

### The one line that differs from self-attention
```python
q = W_q(x_q)     # queries from one sequence
k = W_k(x_kv)    # keys and values from the other
v = W_v(x_kv)
```

That is genuinely all of it — which is the point of doing this one right after
`b_26`. If your `apply_mha` was written without naming the batch axis, this is
almost a rename.

### The trap it adds
`seq_q` and `seq_kv` are **different**. The scores are
`(..., H, seq_q, seq_kv)`, the output length comes from `Q`, and softmax runs
over the last axis (the keys). Anything that assumed a square score matrix
breaks here, and a square test case would not notice.

### A property worth checking yourself
Feed the same array as both inputs and you must get exactly self-attention
back. That single assertion catches most wiring mistakes — a swapped `x_q` /
`x_kv`, or `W_k` fed the wrong sequence.

In [ ]:
# Colab setup (no-op when running locally).
# jax-judge is not published on PyPI, so the judge is installed from the
# repo itself. Regenerate with JAXCODE_REPO=you/YourFork to point this at
# your own fork:  JAXCODE_REPO=you/JAXCode make notebooks
try:
    import google.colab
    get_ipython().run_line_magic('pip', 'install -q flax optax')
    get_ipython().run_line_magic(
        'pip', 'install -q git+https://github.com/YOUR-GITHUB-USERNAME/JAXCode.git')
except ImportError:
    pass

In [ ]:
import jax
import jax.numpy as jnp

print("JAX", jax.__version__, "|", jax.devices())

In [ ]:
# ✅ REFERENCE SOLUTION

import jax
import jax.numpy as jnp


def init_cross_attention(key, d_model, num_heads):
    keys = jax.random.split(key, 4)
    return {
        name: {
            "kernel": jax.random.normal(k, (d_model, d_model)) / jnp.sqrt(d_model),
            "bias": jnp.zeros((d_model,)),
        }
        for name, k in zip(("W_q", "W_k", "W_v", "W_o"), keys)
    }


def _dense(p, x):
    return x @ p["kernel"] + p["bias"]


def apply_cross_attention(params, x_q, x_kv, num_heads):
    d_model = x_q.shape[-1]
    d_k = d_model // num_heads

    def heads(t):
        return t.reshape(*t.shape[:-1], num_heads, d_k).swapaxes(-3, -2)

    # The whole difference from self-attention: q from x_q, k/v from x_kv.
    q = heads(_dense(params["W_q"], x_q))
    k = heads(_dense(params["W_k"], x_kv))
    v = heads(_dense(params["W_v"], x_kv))

    # (..., H, seq_q, seq_kv) — not square, so nothing may assume it is.
    scores = jnp.einsum("...hqd,...hkd->...hqk", q, k) / jnp.sqrt(
        jnp.asarray(d_k, x_q.dtype)
    )
    o = jnp.einsum("...hqk,...hkd->...hqd", jax.nn.softmax(scores, axis=-1), v)

    o = o.swapaxes(-3, -2)
    o = o.reshape(*o.shape[:-2], d_model)
    return _dense(params["W_o"], o)

In [ ]:
# 🔍 Verify
import jax
import jax.numpy as jnp

params = init_cross_attention(jax.random.key(0), d_model=8, num_heads=2)

x_q = jax.random.normal(jax.random.key(1), (2, 3, 8))    # 3 queries
x_kv = jax.random.normal(jax.random.key(2), (2, 7, 8))   # 7 keys/values
print("seq_q=3, seq_kv=7 ->", apply_cross_attention(params, x_q, x_kv, 2).shape)

# Same input twice must reduce to self-attention.
x = jax.random.normal(jax.random.key(3), (2, 5, 8))
same = apply_cross_attention(params, x, x, 2)
print("cross(x, x) shape: ", same.shape)
print("that IS self-attention — the best single check of the wiring")

In [ ]:
# Run the judge against the reference solution
from jax_judge import check

check("cross_attention_pure")